# IMPACT-Polish v2 — baseline TrOCR-pl (segmentacja blla + TrOCR per linia)

Pipeline: blla segmentacja (kraken 7.1.1) -> cropy linii w kolejnosci czytania ->
TrOCR-pl (`PiotrSty/trocr-pl-mixed-v3`) -> scalenie stron -> metryka
`training/transcription_eval.py` (CER/WER + struktura Markdown).

Odniesienie: Tesseract-pol CER 33.11% / WER 81.63% (36 stron, zamrozone test v2).

Wazne: transformers PINE 4.46.3 — wspolzyje z kraken 7.1.1 (safetensors ~=0.7),
nowsze wymagaja safetensors >=0.8 i koliduja z kraken.
Model TrOCR mozna zmienic: PiotrSty/trocr-pl-{base,mixed-v1,mixed-v2,mixed-v3,mixed-aug-light-v1}.

In [ ]:
# 1. Srodowisko: kraken 7.1.1 + transformers 4.46.3 (kompatybilne safetensors)
import subprocess, sys, os
from pathlib import Path
for _w in ('/kaggle/working', '/teamspace/studios/this_studio', '/content', '/workspace'):
    if Path(_w).exists():
        WORKDIR = Path(_w)
        break
else:
    WORKDIR = Path.cwd()
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'transformers'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'kraken==7.1.1', 'jiwer',
                'huggingface_hub'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', 'pillow==11.3.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.46.3'], check=True)
for _mod in list(sys.modules):
    if _mod.startswith(('huggingface_hub.', 'PIL.', 'transformers.')) or _mod in ('huggingface_hub', 'PIL', 'transformers'):
        del sys.modules[_mod]
import torch, kraken, transformers
from importlib.metadata import version as _v
print('WORKDIR:', WORKDIR)
print('kraken', _v('kraken'), '| transformers', _v('transformers'), '| pillow', _v('pillow'),
      '| torch', torch.__version__)
assert torch.cuda.is_available(), 'GPU required'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Bundle benchmarku (test 36 stron) + normalizacja sciezek
import tarfile, json, hashlib
from pathlib import Path
from huggingface_hub import hf_hub_download
BUNDLE = hf_hub_download('PiotrSty/impact-print-v2', 'impact-print-v2-test.tar.gz', repo_type='dataset')
bench = WORKDIR / 'impact-print-v2'
if not (bench / 'test_manifest.jsonl').exists():
    with tarfile.open(BUNDLE) as tar:
        tar.extractall(WORKDIR)
mpath = bench / 'test_manifest.jsonl'
records = [json.loads(l) for l in mpath.read_text(encoding='utf-8').splitlines() if l.strip()]
def _img(r):
    rel = r['image'].replace('\\', '/').split('impact-corpus/')[-1]
    for base in (WORKDIR, bench):
        p = base / 'impact-corpus' / rel
        if p.exists(): return p
        p = base / rel
        if p.exists(): return p
    raise FileNotFoundError(rel)
for r in records:
    p = _img(r)
    assert hashlib.sha256(p.read_bytes()).hexdigest() == r['sha256']
    r['image_path'] = str(p)
print('Test: OK,', len(records), 'stron')

In [ ]:
# 3. Segmentacja blla -> cropy linii (kolejnosc czytania z Segment.lines)
import warnings
from PIL import Image
from kraken.tasks import SegmentationTaskModel
warnings.filterwarnings('ignore')
seg_model = SegmentationTaskModel.load_model()
crops_root = WORKDIR / 'crops'
page_crops = []
for i, r in enumerate(records, 1):
    img = Image.open(r['image_path']).convert('L')
    seg = seg_model.predict(img, __import__('kraken.configs', fromlist=['x']).SegmentationInferenceConfig(accelerator='cuda', device=[0]))
    lines = seg.lines or []
    page_dir = crops_root / r['id']
    page_dir.mkdir(parents=True, exist_ok=True)
    n = 0
    for j, line in enumerate(lines):
        bbox = line.to_bbox() if hasattr(line, 'to_bbox') else line.bbox
        if bbox is None:
            continue
        x0, y0, x1, y1 = (int(v) for v in bbox)
        pad = 4
        crop = img.crop((max(0, x0 - pad), max(0, y0 - pad), x1 + pad, y1 + pad))
        if crop.width < 8 or crop.height < 8:
            continue
        crop.save(page_dir / f'{j:03d}.png')
        n += 1
    page_crops.append({'id': r['id'], 'n': n})
    if i % 10 == 0 or i == len(records):
        print(f'  {i}/{len(records)} stron, linii lacznie:', sum(p['n'] for p in page_crops), flush=True)
print('Crops:', sum(p['n'] for p in page_crops), 'linii na', len(page_crops), 'stron')

In [ ]:
# 4. TrOCR-pl: inferencja na cropach, scalenie per strona
import json, warnings
from pathlib import Path
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch
warnings.filterwarnings('ignore')
MODEL_ID = 'PiotrSty/trocr-pl-mixed-v3'
processor = TrOCRProcessor.from_pretrained(MODEL_ID)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID).cuda().eval()
page_texts = {}
with torch.inference_mode():
    for k, pc in enumerate(page_crops, 1):
        crops = sorted((crops_root / pc['id']).glob('*.png'))
        texts = []
        for c in crops:
            pv = processor(images=c.convert('RGB'), return_tensors='pt').pixel_values.cuda()
            ids = model.generate(pv, max_new_tokens=128)
            texts.append(processor.batch_decode(ids, skip_special_tokens=True)[0].strip())
        page_texts[pc['id']] = '\n'.join(texts)
        if k % 5 == 0 or k == len(page_crops):
            print(f'  {k}/{len(page_crops)} stron', flush=True)
preds_path = WORKDIR / 'impact-print-v2' / 'trocr_pl_mixed_v3_preds.jsonl'
with preds_path.open('w', encoding='utf-8') as fh:
    for r in records:
        fh.write(json.dumps({'id': r['id'], 'status': 'ok', 'text': page_texts[r['id']]},
                            ensure_ascii=False) + '\n')
print('Predictions:', preds_path)

In [ ]:
# 5. Ewaluacja: training.transcription_eval (CER/WER + struktura Markdown)
import json, subprocess, sys
from pathlib import Path
repo = WORKDIR / 'OCR_engine'
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/PiotrStyla/OCR_engine.git', str(repo)], check=True)
bench = WORKDIR / 'impact-print-v2'
r = subprocess.run([sys.executable, '-m', 'training.transcription_eval',
                    '--manifest', str(bench / 'test_manifest.jsonl'),
                    '--predictions', str(preds_path),
                    '--output', str(bench / 'trocr_pl_mixed_v3_v2.json')],
                   cwd=str(repo), capture_output=True, text=True)
print(r.stdout[-1500:] or r.stderr[-1500:])
print('=== ZAPISZ trocr_pl_mixed_v3_v2.json do benchmarks/polocrbench/ results/ ===')

In [ ]:
# 6. Upload wyniku na HF
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault('HF_TOKEN', UserSecretsClient().get_secret('HF_TOKEN'))
except Exception:
    pass
if not os.environ.get('HF_TOKEN'):
    from huggingface_hub import notebook_login
    notebook_login()
from huggingface_hub import upload_file
bench = WORKDIR / 'impact-print-v2'
upload_file(path_or_fileobj=str(bench / 'trocr_pl_mixed_v3_v2.json'),
            path_in_repo='results/trocr_pl_mixed_v3_v2.json',
            repo_id='PiotrSty/impact-print-v2', repo_type='dataset')
print('Uploaded results/trocr_pl_mixed_v3_v2.json')